# Notebook 3 — Evaluate: base vs fine-tuned

The part that makes it engineering. Score the **base** model and your **tuned** model on the SAME
held-out test set, then print a JSON result. **Copy that JSON back into the course** to verify this lab.

Set Runtime -> T4 GPU.


## 1. Install + load both models


In [ ]:
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
from unsloth import FastLanguageModel

def load(name_or_path):
    m, tok = FastLanguageModel.from_pretrained(
        model_name = name_or_path, max_seq_length = 1024, load_in_4bit = True)
    FastLanguageModel.for_inference(m)
    return m, tok

base_model, base_tok = load("unsloth/Llama-3.2-1B-Instruct")
# point this at your saved adapter folder from Notebook 1:
tuned_model, tuned_tok = load("lora_adapter")

## 2. Define a scoreable test set
Each case has an input and a `check` function (objective scoring — the gold standard).
Replace these with YOUR criteria (e.g. exactly 3 bullets, your-voice keywords, etc.).


In [ ]:
test_cases = [
    {"input": "Summarize your week in 3 bullet points.",
     "check": lambda t: t.count("\n-") + t.count("\n*") >= 2},
    {"input": "Reply to a colleague who missed a deadline.",
     "check": lambda t: len(t) > 0},
]

def generate(model, tok, prompt):
    msgs = [{"role":"user","content":prompt}]
    ids = tok.apply_chat_template(msgs, return_tensors="pt",
                                  add_generation_prompt=True).to("cuda")
    out = model.generate(input_ids=ids, max_new_tokens=128, temperature=0.0,
                         do_sample=False)
    return tok.decode(out[0], skip_special_tokens=True)

## 3. Run the eval and print JSON
This prints a JSON object. **Copy the whole line** and paste it into the course's verification box.


In [ ]:
import json

def accuracy(model, tok):
    hits = 0
    for c in test_cases:
        out = generate(model, tok, c["input"])
        hits += 1 if c["check"](out) else 0
    return hits / len(test_cases)

result = {
    "base_accuracy": round(accuracy(base_model, base_tok), 3),
    "tuned_accuracy": round(accuracy(tuned_model, tuned_tok), 3),
    "n_cases": len(test_cases),
}
print(json.dumps(result))

---
If `tuned_accuracy` beat `base_accuracy`, your fine-tune earned its keep. If not, that's a real,
honest result — sometimes the base + a good prompt is enough. Paste the JSON to verify the lab.
